# 00 · Inspect the simulator and check the runtime

The current benchmark forecasts two continuous behavioral coordinates from a known ten-neuron circuit. It holds the tuned selector fixed and compares learned, dense and shuffled support plus direct neural history. A separate preflight seed checks the planted changes, one native fastC-GC fit, and the isolated behavior encoder runtime.

Install the application Python 3.14 environment with `uv sync --python 3.14 --locked --extra notebooks`. From the same repository run `uv sync --project tools/behavior-encoder --python 3.12 --locked` for encoder training. See the [experiment protocol](../docs/effectome_rollout_support.md).


In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select the project Python 3.14 .venv kernel.")

working_directory = Path.cwd().resolve()
try:
    REPO = next(
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "effectome").is_dir()
    )
except StopIteration as error:
    raise RuntimeError("Open this notebook from inside the effectome application repository.") from error

project_src = str(REPO / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

print("Notebook kernel", sys.executable)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
PROJECT_ROOT = REPO


In [ ]:
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from effectome.experiments.behavior_transition_controls import TRANSITION_SCENARIOS, make_transition_control
from effectome.experiments.behavior_transition_graphs import discover_transition_supports, trailing_histories
from effectome.manifold.behavior_dynamics import behavior_runtime_info
from effectome.workflows.cross_method import load_cross_method_profile
from effectome.workflows.resume import content_fingerprint
from synthetic.behavior_rollout_online import GENERATOR
from synthetic.effectome_rollout import READOUT_TRIALS

RUN_SANITY = True
PREFLIGHT_SEED = 90998
print("fastC-GC package", version("fastcgc"))
display(behavior_runtime_info())

conditions = {
    scenario: make_transition_control(GENERATOR, scenario=scenario, seed=PREFLIGHT_SEED)
    for scenario in TRANSITION_SCENARIOS
}
constant, nuisance = conditions["constant"], conditions["nuisance_switch"]
driver = conditions["driver_switch"]
assert all(data.neural.shape[-1] == 10 and data.behavior.shape[-1] == 2 for data in conditions.values())
np.testing.assert_array_equal(constant.mechanisms[0].support, constant.mechanisms[1].support)
assert not np.array_equal(nuisance.mechanisms[0].support, nuisance.mechanisms[1].support)
np.testing.assert_array_equal(constant.behavior, nuisance.behavior)
assert not np.array_equal(driver.behavior, constant.behavior)

fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
for column, (scenario, data) in enumerate(conditions.items()):
    for regime, model in enumerate(data.mechanisms):
        ax = axes[regime, column]
        ax.imshow(model.support, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(f"{scenario.replace('_', ' ').capitalize()} · regime {regime}")
        ax.set(xlabel="Target neuron", ylabel="Source neuron")
plt.show()

if RUN_SANITY:
    _, methods = load_cross_method_profile("csl_sanity", config_root=PROJECT_ROOT / "conf")
    method = methods["fast_cgc"]
    first_training = slice(READOUT_TRIALS, READOUT_TRIALS + 1)
    histories, windows = trailing_histories(
        driver.neural[first_training], driver.origins[first_training], max(GENERATOR.history_lengths),
    )
    signature = content_fingerprint((PREFLIGHT_SEED, histories, windows, method))
    evidence = PROJECT_ROOT / "outputs/effectome-rollout-support/preflight" / f"native-{signature}.json"
    support = discover_transition_supports(histories, windows, method, GENERATOR.fps, evidence, signature)
    assert support.shape == (len(windows), 10, 10)
    display(pd.DataFrame({
        "training_window": np.arange(len(windows)), "estimated_edges": support.sum((1, 2)),
    }))
    print("Native evidence", evidence)


The planted masks describe the simulator. The native fit checks the executable fastC-GC path on three training histories. Continue to notebook 01 for the comparison.
